## Single Responsibility Principle

In [13]:
from abc import ABC, abstractmethod

# zle
class Raport:

    def generate(self):
        print("Generuje raport...")

    def save_to_file(self):
        print("Zapisuje ... ")

# OK
class Raport:
    def generate(self):
        return "Tresc raportu"

class RaportExporter(ABC):

    @abstractmethod
    def export(self):
        print("export ...")


class RaportFileExporter(RaportExporter):

    def __init__(self, file):
        self.file = file

    def export(self, raport):
        print(f"Zapisuje tresc: {raport.generate()} do pliku", self.file)

class RaportEmailExporter(RaportExporter):

    def export(self, raport):
        print(f"Wysyla emaila o tresci: {raport.generate()}")


raport = Raport()
exporters = [RaportFileExporter("nazwa pliku"), RaportEmailExporter()]

for exp in exporters:
    exp.export(raport)

Zapisuje tresc: Tresc raportu do pliku nazwa pliku
Wysyla emaila o tresci: Tresc raportu


## Open / Close


In [11]:
#zle
class Discount:

    def apply_discount(self, price, customer_type):
        if customer_type == "regular":
            return price * 0.9
        elif customer_type == "vip":
            return price * 0.8
        elif customer_type == "super":
            return price * 0.5

In [14]:
# dobrze
class Discount(ABC):

    @abstractmethod
    def apply_discount(self, price):
        pass

class RegularDiscount(Discount):

    def apply_discount(self, price):
        return price * 0.9

class VipDiscount(Discount):

    def apply_discount(self, price):
        return price * 0.8

class SuperDiscount(Discount):

    def apply_discount(self, price):
        return price * 0.5

            

## Liskov Substition Principle

In [16]:
# zle
class Bird:
    def fly(self):
        print("Lecę")

class Pinquin(Bird):
    def fly(self):
        raise NotImplementedError("Pingwin nie lata")

        

In [ ]:
# dobrze

class BirdBase(ABC):
    @abstractmethod
    def move(self): ...

class Falcon(BirdBase):
    def move(self):
        print("Lecę")

class Pinquin(Bird):
    def move(self):
        print("Plynie")

## Interface Segregation Principle

In [18]:
# zle
class Worker:
    def work(self): print("pracuje")

    def eat(self): print("jem")

class Robot(Worker):
    def eat(self): raise NotImplementedError("Roboty nie jedza")
    

In [19]:
# dobrze
class Workable:
    def work(self): print("pracuje")

class Eatable:
    def eat(self): print("jem")

class Human(Workable, Eatable):
    ...

class Robot(Workable):
    ...

In [20]:
## Dependency Inversion Principle


In [ ]:
# Zle

class MySQLDatabse:
    def connect(self):
        print("modul niskopoziomowy - laczenie z baza")

class Application:
    def __init__(self):
        self.db = MySQLDatabse()

    def run(self):
        self.db.connect()
        ...

In [21]:
class Database(ABC):
    @abstractmethod
    def connect(self):
        pass

    @abstractmethod
    def read(self, q):
        pass

    @abstractmethod
    def save(self, data):
        pass

class MySqlDatabaseAdapter(Database):
    def connect(self):
        print("lacze sie z baza mysql...")

    def read(self, q):
        return []

    def save(self, data):
        pass

class PostgresDatabaseAdapter(Database):
    def connect(self):
        print("lacze sie z baza postgres...")

    def read(self, q):
        return []

    def save(self, data):
        pass

class Application:
    def __init__(self, db: Database):
        self.db = db

    def run(self):
        self.db.connect()
        ...


app = Application(db=MySqlDatabaseAdapter())
app.run()


app = Application(db=PostgresDatabaseAdapter())
app.run()


### **Zadanie: Refaktoryzacja kodu pod kątem SOLID**

Masz kod, który działa, ale łamie kilka zasad SOLID. Twoim zadaniem jest poprawienie go tak, aby przestrzegał tych zasad. Kod jest prosty, ale pełen potencjalnych problemów związanych z rozszerzalnością i utrzymaniem.

---

#### **Zastany kod:**
```python
class Order:
    def __init__(self, items, total_price):
        self.items = items
        self.total_price = total_price

    def calculate_discount(self, customer_type):
        if customer_type == "regular":
            return self.total_price * 0.9
        elif customer_type == "vip":
            return self.total_price * 0.8
        else:
            return self.total_price

    def generate_invoice(self):
        invoice = f"Invoice:\nItems: {self.items}\nTotal: {self.total_price}"
        with open("invoice.txt", "w") as file:
            file.write(invoice)
        return invoice

    def send_email(self, email):
        invoice = self.generate_invoice()
        print(f"Sending email to {email} with the following invoice:\n{invoice}")
```

---

#### **Twoje zadanie:**

1. **Zidentyfikuj problemy:**
   - Które zasady SOLID są łamane?
   - Dlaczego kod jest trudny do rozszerzenia i utrzymania?

2. **Zrefaktoryzuj kod:**
   - Zastosuj odpowiednie zasady SOLID.
   - Upewnij się, że kod jest elastyczny, łatwy w testowaniu i zgodny z dobrymi praktykami.

---

#### **Wynik oczekiwany:**

1. Kod powinien być podzielony na mniejsze, bardziej odpowiedzialne klasy.
2. Logika rabatów, generowania faktur i wysyłania e-maili powinna być oddzielona.
3. Nowe typy rabatów (np. „student”) powinny być łatwe do dodania bez modyfikowania istniejącego kodu.





In [29]:
from dataclasses import dataclass

@dataclass
class Order:
    items: list
    total_price: float


class Discount(ABC):

    @abstractmethod
    def apply_discount(self, price):
        pass

class RegularDiscount(Discount):

    def apply_discount(self, price):
        return price * 0.9

class VipDiscount(Discount):

    def apply_discount(self, price):
        return price * 0.8

class SuperDiscount(Discount):

    def apply_discount(self, price):
        return price * 0.5

class InvoiceGenerator:

    def generate_invoice(self, order):
        invoice = f"""Invoice:
        Items: {order.items}
        Total: {order.total_price}"""

        return invoice

class Exporter:
    def export(self, invoice):
        raise NotImplementedError

class InvoiceFileExporter(Exporter):
    def export(self, invoice):
        with open("xxx.txt", "w") as file:
            file.write(invoice)

class InvoiceEmailExporter(Exporter):
    def export(self, invoice):
        print("Wysylam maila z faktura")



class OrderProcessor:

    def __init__(self, discount: Discount, generator: InvoiceGenerator, exporter: Exporter):
        self.discount = discount
        self.generator = generator
        self.exporter = exporter

    def proces_order(self, order: Order):
        order.total_price = self.discount.apply_discount(order.total_price)
        invoice = self.generator.generate_invoice(order)
        self.exporter.export(invoice)
        
order = Order(items=["A"], total_price=100)

discount = VipDiscount()
generator = InvoiceGenerator()
exporter = InvoiceEmailExporter()

processor = OrderProcessor(discount, generator, exporter)
processor.proces_order(order)

Wysylam maila z faktura
